In [1]:
from pathlib import Path
import re 
import numpy as np 
import pandas as pd 

# settings
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

# Paths 
BASE = Path("../../..")
OUT_DIR = BASE / "data" / "output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
def _clean_colnames(cols):
    """Standardize column names: lower-case, strip, replace spaces.punct with _."""
    out = []
    for c in cols:
        c2 = re.sub(r"[^0-9a-zA-Z]+", "_", str(c).strip()).strip("_").lower()
        out.append(c2)
    return out


def read_contract(path: Path) -> pd.DataFrame:
    """Read a single monthly Contract Info file."""
    df = pd.read_csv(path, dtype=str, encoding="latin1")
    df.columns = _clean_colnames(df.columns)

    ren = {
        "contract_id": "contractid",
        "contract":    "contractid",
        "plan_id":     "planid",
        "plan":        "planid",
        "organization_type": "org_type",
        "plan_type":   "plan_type",
        "offers_part_d": "partd",
        "snp_plan":    "snp",
        "organization_name": "org_name",
        "organization_marketing_name": "org_marketing_name",
        "parent_organization": "parent_org",
        "contract_effective_date": "contract_date",
    }
    df = df.rename(columns={k: v for k, v in ren.items() if k in df.columns})

    if "planid" in df.columns:
        df["planid"] = pd.to_numeric(df["planid"], errors="coerce")

    return df

In [3]:
def read_enroll(path: Path) -> pd.DataFrame:
    """Read a single monthly Enrollment Info file."""
    df = pd.read_csv(path, dtype=str, na_values=["*"], encoding="latin1")
    df.columns = _clean_colnames(df.columns)

    ren = {
        "contract_number": "contractid",
        "contract_id":     "contractid",
        "contract":        "contractid",
        "plan_id":         "planid",
        "plan":            "planid",
        "county_name":     "county",
        "state_abbr":      "state",
        "fips_state_county_code": "fips",
        "ssa_state_county_code":  "ssa",
    }
    df = df.rename(columns={k: v for k, v in ren.items() if k in df.columns})

    if "planid" in df.columns:
        df["planid"] = pd.to_numeric(df["planid"], errors="coerce")
    if "fips" in df.columns:
        df["fips"] = pd.to_numeric(df["fips"], errors="coerce")

    return df

In [4]:
def read_service_area(path: Path) -> pd.DataFrame:
    """Read a single monthly service-area file."""
    df = pd.read_csv(path, dtype=str, na_values=["*"], encoding="latin1")
    df.columns = _clean_colnames(df.columns)

    ren = {
        "contract_id":        "contractid",
        "organization_name":  "org_name",
        "organization_type":  "org_type",
    }
    df = df.rename(columns={k: v for k, v in ren.items() if k in df.columns})

    if "partial" in df.columns:
        df["partial"] = df["partial"].map(
            lambda x: str(x).strip().lower() in {"true", "t", "1", "yes", "y"}
            if pd.notna(x) else np.nan
        )
    for col in ["ssa", "fips"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    return df


def load_month(enroll_dir: Path, m: str, y: int) -> pd.DataFrame:
    """Load contract + enrollment for one month and merge them."""
    c_path = enroll_dir / f"CPSC_Contract_Info_{y}_{m}.csv"
    e_path = enroll_dir / f"CPSC_Enrollment_Info_{y}_{m}.csv"

    contract = read_contract(c_path)

    if {"contractid", "planid"}.issubset(contract.columns):
        contract = contract.drop_duplicates(subset=["contractid", "planid"], keep="first")

    enroll = read_enroll(e_path)

    if {"contractid", "planid"}.issubset(enroll.columns) and {"contractid", "planid"}.issubset(contract.columns):
        df = enroll.merge(contract, on=["contractid", "planid"], how="left", suffixes=("", "_contract"))
    else:
        df = enroll.copy()

    df["month"] = int(m)
    df["year"]  = int(y)
    return df


def load_month_sa(sa_dir: Path, m: str, y: int) -> pd.DataFrame:
    """Load one month of service-area data."""
    path = sa_dir / f"MA_Cnty_SA_{y}_{m}.csv"
    df   = read_service_area(path)
    df["month"] = int(m)
    df["year"]  = int(y)
    return df

In [5]:
def build_year_ma(year: int) -> pd.DataFrame:
    print(f"\n{'='*60}")
    print(f"Building MA data for year {year}")
    print(f"{'='*60}")
    
    MONTHS = [f"{m:02d}" for m in range(1, 13)]
    
    enroll_dir = BASE / "data" / "input" / f"enrollment_{year}"
    sa_dir = BASE / "data" / "input" / f"service_area_{year}"
    
    if not enroll_dir.exists():
        print(f"ERROR: Enrollment directory not found: {enroll_dir}")
        return pd.DataFrame()
    if not sa_dir.exists():
        print(f"ERROR: Service area directory not found: {sa_dir}")
        return pd.DataFrame()
    
    print(f"  Reading enrollment data for {year}...")
    plan_year = pd.concat([load_month(enroll_dir, m, year) for m in MONTHS], ignore_index=True)

    sort_cols = [c for c in ["contractid", "planid", "state", "county", "month"] if c in plan_year.columns]
    plan_year = plan_year.sort_values(sort_cols).reset_index(drop=True)
    
    if {"state", "county", "fips"}.issubset(plan_year.columns):
        plan_year["fips"] = (
            plan_year.groupby(["state", "county"], dropna=False)["fips"]
            .transform(lambda s: s.ffill().bfill())
        )
    
    fill_cols_planid = [c for c in ["plan_type", "partd", "snp", "eghp", "plan_name"] if c in plan_year.columns]
    if fill_cols_planid and {"contractid", "planid"}.issubset(plan_year.columns):
        plan_year[fill_cols_planid] = (
            plan_year.groupby(["contractid", "planid"], dropna=False)[fill_cols_planid]
            .transform(lambda df: df.ffill().bfill())
        )

    fill_cols_contract = [c for c in ["org_name", "org_marketing_name", "org_type", "parent_org", "contract_date"] 
                          if c in plan_year.columns]
    if fill_cols_contract and "contractid" in plan_year.columns:
        plan_year[fill_cols_contract] = (
            plan_year.groupby(["contractid"], dropna=False)[fill_cols_contract]
            .transform(lambda df: df.ffill().bfill())
        )
    
    group_cols = [c for c in ["contractid", "planid", "fips", "year"] if c in plan_year.columns]
    plan_year = plan_year.sort_values([*group_cols, "month"]).reset_index(drop=True)
    
    carry_cols = [c for c in [
        "state", "county", "org_type", "plan_type", "partd", "snp", "eghp",
        "org_name", "org_marketing_name", "plan_name", "parent_org", "contract_date",
    ] if c in plan_year.columns]
    
    enroll_like = [c for c in plan_year.columns
                   if any(k in c for k in ["enroll", "enrollment", "benefici", "member", "eligible"])]
    carry_cols += [c for c in enroll_like if c not in carry_cols and c not in group_cols and c != "month"]
    
    final_plans = (
        plan_year.groupby(group_cols, dropna=False, as_index=False)
        .agg({c: "last" for c in carry_cols})
    )
    
    print(f"  Plan data: {len(final_plans):,} rows")

    print(f"  Reading service area data for {year}...")
    service_year = pd.concat([load_month_sa(sa_dir, m, year) for m in MONTHS], ignore_index=True)
    
    sort_cols = [c for c in ["contractid", "fips", "state", "county", "month"] if c in service_year.columns]
    service_year = service_year.sort_values(sort_cols).reset_index(drop=True)
    
    if {"state", "county", "fips"}.issubset(service_year.columns):
        service_year["fips"] = (
            service_year.groupby(["state", "county"], dropna=False)["fips"]
            .transform(lambda s: s.ffill().bfill())
        )
    
    fill_cols_sa = [c for c in ["org_name", "org_type", "plan_type", "partial", "eghp", "ssa", "notes"]
                    if c in service_year.columns]
    if fill_cols_sa and "contractid" in service_year.columns:
        service_year[fill_cols_sa] = (
            service_year.groupby(["contractid"], dropna=False)[fill_cols_sa]
            .transform(lambda df: df.ffill().bfill())
        )
    
    group_cols_sa = [c for c in ["contractid", "fips", "year"] if c in service_year.columns]
    service_year = service_year.sort_values([*group_cols_sa, "month"]).reset_index(drop=True)
    
    carry_cols_sa = [c for c in ["state", "county", "org_name", "org_type", "plan_type",
                                  "partial", "eghp", "ssa", "notes"]
                     if c in service_year.columns]
    
    final_service_area = (
        service_year.groupby(group_cols_sa, dropna=False, as_index=False)
        .agg({c: "last" for c in carry_cols_sa})
    )
    
    print(f"  Service area data: {len(final_service_area):,} rows")
    
    print(f"  Merging plan and service area data...")
    merged = final_plans.merge(
        final_service_area,
        on=["contractid", "fips", "year"],
        how="inner",
        suffixes=("_plan", "_sa"),
    )
    
    print(f"  Merged data (before filters): {len(merged):,} rows")
    
    planid_num = pd.to_numeric(merged.get("planid"), errors="coerce")
    
    snp_col = "snp" if "snp" in merged.columns else None
    plan_type_col = "plan_type_plan" if "plan_type_plan" in merged.columns else "plan_type"
    
    filtered = merged.copy()
    
    if snp_col:
        filtered = filtered[filtered[snp_col] == "No"]
        print(f"  After removing SNPs: {len(filtered):,} rows")
    
    filtered = filtered[(planid_num < 800) | (planid_num >= 900)]
    print(f"  After removing 800-series: {len(filtered):,} rows")
    
    if plan_type_col in filtered.columns:
        filtered["is_pdp_only"] = filtered[plan_type_col].astype(str).str.contains(
            r"(?i)(prescription|^pdp|drug plan)", 
            na=False, 
            regex=True
        ) & ~filtered[plan_type_col].astype(str).str.contains(
            r"(?i)(ma-pd|hmo|ppo|pffs|msa)", 
            na=False, 
            regex=True
        )
        filtered = filtered[~filtered["is_pdp_only"]]
        filtered = filtered.drop(columns=["is_pdp_only"])
        print(f"  After removing drug-only plans: {len(filtered):,} rows")
    
    print(f"  Final filtered data: {len(filtered):,} rows")
    
    out_path = OUT_DIR / f"ma_data_{year}.csv"
    filtered.to_csv(out_path, index=False)
    print(f"  Saved to: {out_path}")
    
    return filtered

In [6]:
def build_all_years(years: list = [2010, 2011, 2012, 2013, 2014, 2015], save: bool = True):
    results = {}
    
    for year in years:
        try:
            df = build_year_ma(year)
            results[year] = df
        except Exception as e:
            print(f"\n✗ ERROR building data for {year}: {e}")
            results[year] = pd.DataFrame()
    
    print(f"\n{'='*60}")
    print("Summary:")
    print(f"{'='*60}")
    for year, df in results.items():
        status = "✓" if not df.empty else "✗"
        print(f"{status} {year}: {len(df):,} rows")
    
    return results

In [7]:
# Build all years 2010–2015
results = build_all_years(years=[2010, 2011, 2012, 2013, 2014, 2015])


Building MA data for year 2010
  Reading enrollment data for 2010...


KeyboardInterrupt: 

In [ ]:
# Run this to compare plan types and SNP filtering across years
for year in [2010, 2011, 2012]:
    df = results[year]
    print(f"\n--- {year} ({len(df):,} rows) ---")
    if 'snp' in df.columns:
        print("SNP values:", df['snp'].value_counts().to_dict())
    pt_col = 'plan_type_plan' if 'plan_type_plan' in df.columns else 'plan_type'
    if pt_col in df.columns:
        print("Plan types:", df[pt_col].value_counts().to_dict())
    if 'planid' in df.columns:
        import pandas as pd
        planid_num = pd.to_numeric(df['planid'], errors='coerce')
        print("800-series plans:", ((planid_num >= 800) & (planid_num < 900)).sum())


--- 2010 (112,856 rows) ---
SNP values: {'No': 112856}
Plan types: {'PFFS': 63230, 'HMO/HMOPOS': 22485, 'Local PPO': 12120, 'Regional PPO': 10601, '1876 Cost': 3814, 'National PACE': 376, 'ESRD I': 111, 'MSA': 69, 'Continuing Care Retirement Community': 36, 'PSO (State License)': 14}
800-series plans: 0

--- 2011 (70,184 rows) ---
SNP values: {'No': 70184}
Plan types: {'HMO/HMOPOS': 20337, 'PFFS': 18476, 'Local PPO': 15294, 'Regional PPO': 10720, '1876 Cost': 4777, 'National PACE': 435, 'MSA': 133, 'PSO (State License)': 12}
800-series plans: 0

--- 2012 (69,770 rows) ---
SNP values: {'No': 69770}
Plan types: {'HMO/HMOPOS': 20797, 'Local PPO': 17696, 'PFFS': 14800, 'Regional PPO': 10408, '1876 Cost': 5425, 'National PACE': 492, 'MSA': 142, 'PSO (State License)': 10}
800-series plans: 0


In [ ]:
df = pd.read_csv("../../../data/output/ma_data_2010.csv")
enroll_cols = [c for c in df.columns if any(k in c for k in ["enroll", "enrollment", "benefici", "member"])]
print(enroll_cols)

['enrollment']


/var/folders/dr/bz36x93s5vs6lnjjq8z60zfm0000gp/T/ipykernel_37377/3462753771.py:1: DtypeWarning: Columns (25) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../../../data/output/ma_data_2010.csv")
